In [1]:
import os
import pickle

import pandas as pd
import mlflow
import xgboost as xgb

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/opt/conda/envs/exp-tracking-env/lib/python3.9/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Configure MLflow tracking

This section tells the notebook where to save MLflow experiment information.

The notebook writes experiment data to `mlflow.db`, and the MLflow UI reads from the same database file.

In [2]:
# Define where MLflow should store experiment tracking data.
# We use an absolute path so the notebook and MLflow UI use the same database.
MLFLOW_TRACKING_URI = "sqlite:////workspaces/mlops/mlflow.db"

# Define the experiment name.
# All runs from this notebook will be grouped under this experiment in the MLflow UI.
EXPERIMENT_NAME = "nyc-taxi-experiment"

# Tell the notebook/Python code where to write MLflow data.
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Create or select the experiment.
mlflow.set_experiment(EXPERIMENT_NAME)

# Print values to confirm the configuration.
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

2026/06/08 07:00:27 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/08 07:00:27 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


Tracking URI: sqlite:////workspaces/mlops/mlflow.db
Experiment: nyc-taxi-experiment


## 3. Load taxi data

We use NYC yellow taxi trip data.

For training, we use January 2024.
For validation, we use February 2024.

The helper function reads a parquet file, calculates trip duration in minutes, removes unrealistic durations, and prepares location IDs as categorical values.

In [3]:
# URLs for NYC yellow taxi data.
# January is used for training, February for validation.
TRAIN_DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
VALID_DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-02.parquet"

# Columns we need for this project.
# Reading only necessary columns reduces memory usage.
COLUMNS = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]


def read_dataframe(filename, sample_size=None):
    """
    Read taxi data and prepare the target variable: trip duration in minutes.
    
    Parameters
    ----------
    filename : str
        Local path or URL to a parquet file.
    sample_size : int or None
        If provided, keep only a random sample of rows to avoid memory problems.
    
    Returns
    -------
    pandas.DataFrame
        Cleaned dataframe with a duration column.
    """
    
    # Read selected columns from the parquet file.
    df = pd.read_parquet(filename, columns=COLUMNS)

    # Optional: use a smaller random sample for faster development.
    if sample_size is not None:
        df = df.sample(n=sample_size, random_state=42)

    # Calculate trip duration as dropoff time minus pickup time.
    df["duration"] = df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]

    # Convert duration from timedelta to minutes.
    # This vectorized version is faster than .apply(lambda ...).
    df["duration"] = df["duration"].dt.total_seconds() / 60

    # Keep only reasonable trip durations.
    df = df[(df["duration"] >= 1) & (df["duration"] <= 60)]

    # Treat location IDs as categories, not numbers.
    categorical = ["PULocationID", "DOLocationID"]
    df[categorical] = df[categorical].astype(str)

    return df

## 4. Read training and validation data

We load January 2024 as training data and February 2024 as validation data.

To avoid memory problems while developing in Codespaces, we start with a small sample of each month.

In [4]:
# Use a small sample first so the notebook runs faster and does not crash.
SAMPLE_SIZE = 50_000

# Read and clean training data.
df_train = read_dataframe(TRAIN_DATA_URL, sample_size=SAMPLE_SIZE)

# Read and clean validation data.
df_val = read_dataframe(VALID_DATA_URL, sample_size=SAMPLE_SIZE)

# Show the number of rows and columns after cleaning.
print("Training shape:", df_train.shape)
print("Validation shape:", df_val.shape)

# Preview the training dataframe.
df_train.head()

Training shape: (48866, 6)
Validation shape: (48883, 6)


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,duration
1725696,2024-01-20 13:31:30,2024-01-20 14:03:25,132,233,17.14,31.916667
1581136,2024-01-18 21:52:46,2024-01-18 22:03:21,163,75,2.49,10.583333
19137,2024-01-01 03:43:58,2024-01-01 03:50:47,127,20,1.84,6.816667
1682810,2024-01-19 22:20:12,2024-01-19 22:50:12,186,263,3.60,30.000000
511035,2024-01-06 22:41:50,2024-01-06 22:43:24,238,238,0.04,1.566667


## 5. Feature engineering and vectorization

Machine learning models need numeric input.

We create a route feature called `PU_DO`, which combines pickup and dropoff location IDs.

Then we use `DictVectorizer` to convert categorical features into one-hot encoded columns and keep numerical features as numbers.

In [5]:
# Create a route feature by combining pickup and dropoff location IDs.
# Example: pickup 161 and dropoff 236 becomes "161_236".
df_train["PU_DO"] = df_train["PULocationID"] + "_" + df_train["DOLocationID"]
df_val["PU_DO"] = df_val["PULocationID"] + "_" + df_val["DOLocationID"]

# Define which columns are categorical and which are numerical.
categorical = ["PU_DO"]
numerical = ["trip_distance"]

# Convert selected feature columns into dictionaries.
# DictVectorizer expects data in this format:
# [{"PU_DO": "161_236", "trip_distance": 2.5}, ...]
train_dicts = df_train[categorical + numerical].to_dict(orient="records")
val_dicts = df_val[categorical + numerical].to_dict(orient="records")

# Create the vectorizer.
dv = DictVectorizer()

# Fit on training data and transform training data.
X_train = dv.fit_transform(train_dicts)

# Transform validation data using the same vectorizer.
X_val = dv.transform(val_dicts)

# Define target values: trip duration in minutes.
y_train = df_train["duration"].values
y_val = df_val["duration"].values

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("Number of target values:", len(y_train), len(y_val))

X_train shape: (48866, 5239)
X_val shape: (48883, 5239)
Number of target values: 48866 48883


## 6. Train a baseline Linear Regression model

Before using a more complex model like XGBoost, we train a simple Linear Regression model.

This gives us a baseline RMSE. Later, XGBoost should ideally perform better than this baseline.

In [6]:
# Create a simple baseline regression model.
lr = LinearRegression()

# Train the model on the training feature matrix and target values.
lr.fit(X_train, y_train)

# Predict trip duration on the validation set.
y_pred = lr.predict(X_val)

# Calculate validation RMSE.
# RMSE is measured in minutes because the target variable is duration in minutes.
rmse = root_mean_squared_error(y_val, y_pred)

print("Linear Regression validation RMSE:", rmse)

Linear Regression validation RMSE: 5.807450769370368


## 7. Log the baseline model to MLflow

Now we train the Linear Regression baseline again inside an MLflow run.

Inside the run, we log:
- the model type
- the validation RMSE
- the trained Linear Regression model

After running this cell, refresh the MLflow UI and you should see a new run.

In [7]:
# Start a new MLflow run for the baseline model.
with mlflow.start_run(run_name="linear-regression-baseline"):

    # Add a tag so we can identify the model type in the MLflow UI.
    mlflow.set_tag("model", "linear_regression")

    # Train the baseline model.
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    # Predict on validation data.
    y_pred = lr.predict(X_val)

    # Calculate validation RMSE.
    rmse = root_mean_squared_error(y_val, y_pred)

    # Log the RMSE metric to MLflow.
    mlflow.log_metric("rmse", rmse)

    # Log the trained sklearn model to MLflow.
    mlflow.sklearn.log_model(
        sk_model=lr,
        name="linear_regression_model"
    )

    print("Logged Linear Regression RMSE:", rmse)

2026/06/08 07:00:36 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logged Linear Regression RMSE: 5.807450769370368


## 8. Prepare XGBoost data

XGBoost uses a special optimized data structure called `DMatrix`.

We convert the training and validation feature matrices into `DMatrix` objects before training an XGBoost model.

In [8]:
# Convert the training data into XGBoost's optimized DMatrix format.
train = xgb.DMatrix(X_train, label=y_train)

# Convert the validation data into XGBoost's optimized DMatrix format.
valid = xgb.DMatrix(X_val, label=y_val)

print("XGBoost training data is ready.")
print("Training rows:", X_train.shape[0])
print("Validation rows:", X_val.shape[0])

XGBoost training data is ready.
Training rows: 48866
Validation rows: 48883


## 9. Define the Hyperopt objective function

We use Hyperopt to search for good XGBoost hyperparameters.

For each trial, we log to MLflow:

- hyperparameters
- validation RMSE
- training duration in seconds
- model size in MB
- trial number

This helps us choose a model not only by accuracy, but also by speed and size.

In [11]:
import time
import tempfile
from pathlib import Path

# Counter used to give each Hyperopt trial a unique MLflow run name.
trial_counter = 0


def objective(params):
    """
    Train and evaluate one XGBoost model using one set of hyperparameters.

    Hyperopt calls this function many times.
    Each call is one trial.
    Each trial is logged as a separate MLflow run.
    """

    global trial_counter
    trial_counter += 1

    with mlflow.start_run(run_name=f"xgboost-hyperopt-trial-{trial_counter}"):

        mlflow.set_tag("model", "xgboost")
        mlflow.set_tag("run_type", "hyperopt_trial")
        mlflow.set_tag("trial_number", trial_counter)

        mlflow.log_params(params)

        start_time = time.time()

        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=100,
            evals=[(valid, "validation")],
            early_stopping_rounds=20,
            verbose_eval=False,
        )

        duration_seconds = time.time() - start_time

        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)

        # Save the model temporarily to estimate its size.
        with tempfile.TemporaryDirectory() as tmp_dir:
            model_path = Path(tmp_dir) / "model.json"
            booster.save_model(model_path)
            model_size_mb = model_path.stat().st_size / (1024 * 1024)

        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("training_duration_seconds", duration_seconds)
        mlflow.log_metric("model_size_mb", model_size_mb)

        return {
            "loss": rmse,
            "status": STATUS_OK,
        }

## 10. Run hyperparameter tuning with Hyperopt

Now we define the hyperparameter search space.

Hyperopt will choose different values from this space and pass them to the `objective()` function.

For each trial:
- one XGBoost model is trained
- parameters are logged to MLflow
- RMSE is logged to MLflow
- Hyperopt uses RMSE to decide which parameters to try next

We start with a small number of trials to keep the notebook fast.

In [15]:
# Define the hyperparameter search space for XGBoost.
search_space = {
    # Tree depth must be an integer.
    "max_depth": scope.int(hp.quniform("max_depth", 4, 20, 1)),

    # Learning rate controls how much each tree contributes.
    "learning_rate": hp.loguniform("learning_rate", -3, 0),

    # Minimum child weight controls how conservative tree splitting is.
    "min_child_weight": hp.loguniform("min_child_weight", -1, 3),

    # L1 regularization.
    "reg_alpha": hp.loguniform("reg_alpha", -5, -1),

    # L2 regularization.
    "reg_lambda": hp.loguniform("reg_lambda", -6, -1),

    # Regression objective.
    "objective": "reg:squarederror",

    # Random seed for reproducibility.
    "seed": 42,
}

# Store results from all Hyperopt trials.
trials = Trials()

# Run hyperparameter optimization.
# max_evals controls how many different parameter combinations Hyperopt tries.
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=10,
    trials=trials,
)

print("Best hyperparameters:")
print(best_result)

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:38<00:00,  3.83s/trial, best loss: 5.426424467491074]
Best hyperparameters:
{'learning_rate': np.float64(0.4418668546485012), 'max_depth': np.float64(15.0), 'min_child_weight': np.float64(4.10011407578665), 'reg_alpha': np.float64(0.17885326474021432), 'reg_lambda': np.float64(0.005803881670069494)}


## 11. Compare good Hyperopt runs

Now we query MLflow and keep only Hyperopt trial runs with RMSE below 5.6.

We display them in a dataframe with:

- run name
- run ID
- RMSE
- training duration
- model size
- hyperparameters

This helps us choose a model based on accuracy, speed, and size.

In [16]:
from mlflow.tracking import MlflowClient

# Connect to the same MLflow tracking database.
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

# Get the experiment by name.
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

# Search only Hyperopt trial runs with RMSE below 5.6.
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_type = 'hyperopt_trial' and metrics.rmse < 5.52",
    order_by=["metrics.rmse ASC"],
)

# Collect run information into a list of dictionaries.
run_data = []

for run in runs:
    metrics = run.data.metrics
    params = run.data.params

    run_data.append({
        "run_name": run.info.run_name,
        "run_id": run.info.run_id,
        "rmse": metrics.get("rmse"),
        "training_duration_seconds": metrics.get("training_duration_seconds"),
        "model_size_mb": metrics.get("model_size_mb"),
        "learning_rate": params.get("learning_rate"),
        "max_depth": params.get("max_depth"),
        "min_child_weight": params.get("min_child_weight"),
        "reg_alpha": params.get("reg_alpha"),
        "reg_lambda": params.get("reg_lambda"),
    })

# Convert to dataframe.
good_runs_df = pd.DataFrame(run_data)

# Sort by RMSE from best to worst.
good_runs_df = good_runs_df.sort_values("rmse")

good_runs_df

,run_name,run_id,rmse,training_duration_seconds,model_size_mb,learning_rate,max_depth,min_child_weight,reg_alpha,reg_lambda
0,xgboost-hyperopt-trial-9,d0f29876ceec4b2ea280e6346ac42e02,5.426424,3.190955,0.634630,0.4418668546485012,15,4.10011407578665,0.17885326474021432,0.005803881670069494
1,xgboost-hyperopt-trial-7,443bc6b7e37648d98acd794ec3d11d14,5.442394,5.439653,1.182989,0.18171463660429502,19,5.560892207242158,0.03280410706866543,0.003666690408977374
2,xgboost-hyperopt-trial-10,f73d8af9168149389e8a722c929d3945,5.445345,1.964868,0.283576,0.5563841291807892,8,4.318621541567496,0.010122017193764064,0.16766917233794348
3,xgboost-hyperopt-trial-1,8914bb603fd24cf3af1a5a6820d1c4ec,5.474738,2.534137,0.502023,0.22902800071024557,11,6.172880737551628,0.007813515929771676,0.06260368156271655
4,xgboost-hyperopt-trial-3,a261972b5b6e4b0d8edf6423cbc2ab3c,5.475778,7.220327,1.741864,0.16729330996600225,19,1.350640237417495,0.007857974375983208,0.1982807296849069
5,xgboost-hyperopt-trial-2,6d93038dcc674de7977b3bff67b645c5,5.501220,1.931163,0.270933,0.8233665697338255,11,16.530119152926776,0.07452711491554115,0.2702905159415732
6,xgboost-hyperopt-trial-6,d5982393a74f4786839590416dbede28,5.504121,NaN,NaN,0.5154839835958396,6,10.599994048343314,0.0112360604030843,0.028198769763146156
7,xgboost-hyperopt-trial-3,38cc48fd256a44c9a67b4e2b3374ee9d,5.512629,NaN,NaN,0.36849457641428285,5,5.789878746687666,0.12379388866890788,0.025987428954045863
8,xgboost-hyperopt-trial-1,d606aeba3f544aaf84eb02e8275a69ff,5.514055,NaN,NaN,0.3156666978775286,16,19.457769320394213,0.023394218124174363,0.010609488892262098
9,xgboost-hyperopt-trial-2,5f5d66327c184b5688d53ed7f94d5838,5.515836,NaN,NaN,0.08119009972280274,13,1.470508944669837,0.010702300927552384,0.0040335603419135255


## 12. Select the best Hyperopt run

Now we select the best XGBoost trial from MLflow.

We search all Hyperopt trial runs, sort them by RMSE from lowest to highest, and choose the first one.

This gives us:
- the best run ID
- the best RMSE
- the best hyperparameters logged during tuning

In [17]:
# Search all Hyperopt trial runs and sort by RMSE from lowest to highest.
best_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_type = 'hyperopt_trial'",
    order_by=["metrics.rmse ASC"],
    max_results=1,
)

# Select the best run.
best_run = best_runs[0]

# Print important information about the best run.
print("Best run name:", best_run.info.run_name)
print("Best run ID:", best_run.info.run_id)
print("Best RMSE:", best_run.data.metrics["rmse"])
print("Best parameters:")
print(best_run.data.params)

Best run name: xgboost-hyperopt-trial-9
Best run ID: d0f29876ceec4b2ea280e6346ac42e02
Best RMSE: 5.426424467491074
Best parameters:
{'learning_rate': '0.4418668546485012', 'max_depth': '15', 'min_child_weight': '4.10011407578665', 'objective': 'reg:squarederror', 'reg_alpha': '0.17885326474021432', 'reg_lambda': '0.005803881670069494', 'seed': '42'}


## 13. Prepare the final XGBoost parameters

MLflow stores parameters as strings.

Before using the best parameters to train the final XGBoost model, we convert them back to the correct Python types:

- `learning_rate` → float
- `max_depth` → int
- `min_child_weight` → float
- `reg_alpha` → float
- `reg_lambda` → float

We also add fixed XGBoost parameters like `objective` and `seed`.

In [19]:
# Convert the best run parameters from strings to proper Python types.
best_params = {
    "learning_rate": float(best_run.data.params["learning_rate"]),
    "max_depth": int(best_run.data.params["max_depth"]),
    "min_child_weight": float(best_run.data.params["min_child_weight"]),
    "reg_alpha": float(best_run.data.params["reg_alpha"]),
    "reg_lambda": float(best_run.data.params["reg_lambda"]),
    "objective": "reg:squarederror",
    "seed": 42,
}

print("Final XGBoost parameters:")
print(best_params)

Final XGBoost parameters:
{'learning_rate': 0.4418668546485012, 'max_depth': 15, 'min_child_weight': 4.10011407578665, 'reg_alpha': 0.17885326474021432, 'reg_lambda': 0.005803881670069494, 'objective': 'reg:squarederror', 'seed': 42}


## 14. Train and log the final XGBoost model

Now we train the final XGBoost model using the best hyperparameters selected from MLflow.

Inside this final MLflow run, we log:

- the final XGBoost parameters
- the validation RMSE
- the trained XGBoost model
- the `DictVectorizer` object used for preprocessing

Saving the preprocessor is important because the model expects input data to be transformed in exactly the same way during future prediction.

In [30]:
# Create a local folder for temporary model/preprocessor files.
# This folder is ignored by Git because it is listed in .gitignore.
os.makedirs("models", exist_ok=True)

# Path where we temporarily save the DictVectorizer.
preprocessor_path = "models/preprocessor.b"

# Start a new MLflow run for the final XGBoost model.
with mlflow.start_run(run_name="xgboost-final-model"):

    # Add tags to identify this as the final selected model.
    mlflow.set_tag("model", "xgboost")
    mlflow.set_tag("run_type", "final_model")
    mlflow.set_tag("selected_from_run_id", best_run.info.run_id)

    # Log the final XGBoost parameters.
    mlflow.log_params(best_params)

    # Log data information.
    # We do not log the full raw data because it is large.
    # Instead, we log enough metadata to know which data was used.
    mlflow.log_param("train_data_url", TRAIN_DATA_URL)
    mlflow.log_param("valid_data_url", VALID_DATA_URL)
    mlflow.log_param("sample_size", SAMPLE_SIZE)

    mlflow.log_metric("train_rows", df_train.shape[0])
    mlflow.log_metric("val_rows", df_val.shape[0])
    mlflow.log_metric("num_features", X_train.shape[1])

    # Train the final XGBoost model using the best parameters.
    final_booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=100,
        evals=[(valid, "validation")],
        early_stopping_rounds=20,
        verbose_eval=10,
    )

    # Predict on validation data.
    y_pred = final_booster.predict(valid)

    # Calculate validation RMSE.
    final_rmse = root_mean_squared_error(y_val, y_pred)

    # Log final RMSE to MLflow.
    mlflow.log_metric("rmse", final_rmse)

    # Save the fitted DictVectorizer locally.
    with open(preprocessor_path, "wb") as f_out:
        pickle.dump(dv, f_out)

    # Log the preprocessor file as an MLflow artifact.
    mlflow.log_artifact(preprocessor_path, artifact_path="preprocessor")

    # Log the final XGBoost model to MLflow.
    mlflow.xgboost.log_model(
        xgb_model=final_booster,
        name="xgboost_model"
    )

    print("Final XGBoost RMSE:", final_rmse)

[0]	validation-rmse:7.52093
[10]	validation-rmse:5.50577
[20]	validation-rmse:5.48158
[30]	validation-rmse:5.46829
[40]	validation-rmse:5.45820
[50]	validation-rmse:5.45008
[60]	validation-rmse:5.44400
[70]	validation-rmse:5.43968
[80]	validation-rmse:5.43232
[90]	validation-rmse:5.43008
[99]	validation-rmse:5.42642


/opt/conda/envs/exp-tracking-env/lib/python3.9/site-packages/xgboost/core.py:158: UserWarning: [08:25:31] WARNING: /workspace/src/c_api/c_api.cc:1374: Saving model in the UBJSON format as default.  You can use file extension: `json`, `ubj` or `deprecated` to choose between formats.
  warnings.warn(smsg, UserWarning)
2026/06/08 08:25:38 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Final XGBoost RMSE: 5.426424467491074


## 15. Compare the final XGBoost model with the Linear Regression baseline

MLflow search filters do not support `or` in this version.

So we first get all runs from the experiment, then filter the results in pandas to keep only:

- `linear-regression-baseline`
- `xgboost-final-model`

Then we compare their RMSE values.

In [22]:
# Get all runs from the experiment, sorted by RMSE.
all_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.rmse ASC"],
)

# Collect run information.
comparison_data = []

for run in all_runs:
    comparison_data.append({
        "run_name": run.info.run_name,
        "run_id": run.info.run_id,
        "model": run.data.tags.get("model"),
        "run_type": run.data.tags.get("run_type"),
        "rmse": run.data.metrics.get("rmse"),
    })

# Create dataframe.
comparison_df = pd.DataFrame(comparison_data)

# Keep only the baseline and final model runs.
comparison_df = comparison_df[
    comparison_df["run_name"].isin([
        "linear-regression-baseline",
        "xgboost-final-model",
    ])
]

# Sort by RMSE.
comparison_df = comparison_df.sort_values("rmse")

comparison_df

,run_name,run_id,model,run_type,rmse
0,xgboost-final-model,ac7ac025b36a4d358e8903418e06bc24,xgboost,final_model,5.426424
29,linear-regression-baseline,28f7105fafc142aba6936c9f78391693,linear_regression,None,5.807451
30,linear-regression-baseline,a03abcad819c461aad0d67c55eebfa90,linear_regression,None,5.807451


## 16. Archive existing model versions

Before promoting the final XGBoost model, we archive older versions of the registered model.

Archived versions are not deleted. They are kept for history, but they are no longer considered active.

In this project, we already have older registered versions of `nyc-taxi-regressor`, so we move them to the `Archived` stage before registering and promoting the final XGBoost model.

In [23]:
model_name = "nyc-taxi-regressor"

# Get all existing versions of the registered model.
model_versions = client.search_model_versions(f"name = '{model_name}'")

# Move all existing versions to Archived.
for version in model_versions:
    client.transition_model_version_stage(
        name=model_name,
        version=version.version,
        stage="Archived",
        archive_existing_versions=False,
    )

    print(f"Archived version {version.version}")

2026/06/08 08:05:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/08 08:05:05 INFO mlflow.store.db.utils: Updating database tables
INFO  [alembic.runtime.migration] Context impl SQLiteImpl.
INFO  [alembic.runtime.migration] Will assume non-transactional DDL.


## 17. Register the final XGBoost model

Now we register the final XGBoost model in the MLflow Model Registry.

The final model was logged inside the `xgboost-final-model` run with artifact name `xgboost_model`.

Registering the model creates a new version under the registered model name `nyc-taxi-regressor`.

In [31]:
# Find the latest final XGBoost model run.
final_model_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.run_name = 'xgboost-final-model'",
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

# Select the latest final model run.
final_model_run = final_model_runs[0]

print("Final model run ID:", final_model_run.info.run_id)
print("Final model RMSE:", final_model_run.data.metrics["rmse"])

# Build the model URI.
# This points to the xgboost_model artifact inside the final model run.
model_uri = f"runs:/{final_model_run.info.run_id}/xgboost_model"

print("Model URI:", model_uri)

# Register the final XGBoost model.
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=model_name,
)

print("Registered model name:", registered_model.name)
print("Registered model version:", registered_model.version)

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
2026/06/08 08:26:16 WARNING mlflow.tracking._model_registry.fluent: Run with id 829fd02e9607490c8c1a526576f791c3 has no artifacts at artifact path 'xgboost_model', registering model based on models:/m-10460775a89a49519623ad33585f39dd instead


Final model run ID: 829fd02e9607490c8c1a526576f791c3
Final model RMSE: 5.426424467491074
Model URI: runs:/829fd02e9607490c8c1a526576f791c3/xgboost_model
Registered model name: nyc-taxi-regressor
Registered model version: 2


Created version '2' of model 'nyc-taxi-regressor'.


## 18. Move the final model to Staging

Now we move the newly registered XGBoost model version to the `Staging` stage.

`Staging` means this model version is selected as a candidate for testing before Production.

We use `archive_existing_versions=True` so that if another model version was already in Staging, MLflow archives it automatically.

In [32]:
# Move the newly registered model version to Staging.
client.transition_model_version_stage(
    name=model_name,
    version=registered_model.version,
    stage="Staging",
    archive_existing_versions=True,
)

# Add a description to this model version.
client.update_model_version(
    name=model_name,
    version=registered_model.version,
    description=(
        "Final XGBoost model trained on NYC yellow taxi data. "
        "Selected based on lowest validation RMSE from Hyperopt tuning. "
        f"Source run ID: {final_model_run.info.run_id}. "
        f"Validation RMSE: {final_model_run.data.metrics['rmse']:.4f}."
    )
)

print(f"Moved model {model_name} version {registered_model.version} to Staging")

Moved model nyc-taxi-regressor version 2 to Staging


/tmp/ipykernel_6923/3594269441.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


In [33]:
client.update_registered_model(
    name=model_name,
    description=(
        "NYC taxi trip duration prediction model. "
        "Model versions are promoted through Staging and Production."
    )
)

<RegisteredModel: aliases={}, creation_timestamp=1780905992676, deployment_job_id=None, deployment_job_state=None, description=('NYC taxi trip duration prediction model. Model versions are promoted through '
 'Staging and Production.'), last_updated_timestamp=1780907182985, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1780905992727, current_stage='Archived', deployment_job_state=None, description=('Final XGBoost model trained on NYC yellow taxi data. Selected based on '
 'lowest validation RMSE from Hyperopt tuning. Source run ID: '
 'ac7ac025b36a4d358e8903418e06bc24. Validation RMSE: 5.4264.'), last_updated_timestamp=1780907179702, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='ac7ac025b36a4d358e8903418e06bc24', run_link=None, source='models:/m-9698cb445de44b0e97b792b29ac85d67', status='READY', status_message=None, tags={}, user_id=None, version=1>,
 <ModelVersion: aliases=[], creation_timestamp=1780907176199, current_stage='Staging', de

## 19. Check registered model versions and stages

We check the registered model versions directly from MLflow using `MlflowClient`.

This lets us see each version and its current stage, such as:

- None
- Staging
- Production
- Archived

In [34]:
model_versions = client.search_model_versions(f"name = '{model_name}'")

for version in model_versions:
    print(
        f"Version: {version.version}, "
        f"Stage: {version.current_stage}, "
        f"Run ID: {version.run_id}"
    )

Version: 2, Stage: Staging, Run ID: 829fd02e9607490c8c1a526576f791c3
Version: 1, Stage: Archived, Run ID: ac7ac025b36a4d358e8903418e06bc24


## Delete an extra registered model

We accidentally created two registered model names:

- `nyc-taxi-regressor`
- `nyc_taxi_regressor`

We want to keep `nyc-taxi-regressor` and delete the extra one: `nyc_taxi_regressor`.

Deleting a registered model removes it from the MLflow Model Registry, including its registered versions.

This does not delete the original MLflow experiment runs.

In [29]:
from mlflow.exceptions import MlflowException

# This is the extra registered model name we want to delete.
# We keep "nyc-taxi-regressor" and delete "nyc_taxi_regressor".
model_to_delete = "nyc_taxi_regressor"

try:
    # Delete the registered model from the MLflow Model Registry.
    # This removes the model entry and its registered versions.
    client.delete_registered_model(name=model_to_delete)

    print(f"Deleted registered model: {model_to_delete}")

except MlflowException as e:
    # If the model does not exist or cannot be deleted,
    # show a friendly message instead of crashing the notebook.
    print(f"Could not delete registered model: {model_to_delete}")
    print(e)

Deleted registered model: nyc_taxi_regressor
